In [ ]:
import json
import os
import subprocess
from concurrent.futures import ThreadPoolExecutor, as_completed
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import plotly.express as px
import pybigtools
from tqdm.auto import tqdm

In [2]:
bw_dir = Path(
    "/project/milne_group/asmith/Projects/2025-07-19-myeloid-specific-enhancer-identification/data/processed/wimm_atlas/bigwigs/"
)
bw_files = list(bw_dir.rglob("*.bw")) + list(bw_dir.rglob("*.bigWig"))
print(f"Found {len(bw_files)} bigwig files")

Found 934 bigwig files


In [3]:
tss_scores = pd.read_parquet(
    "/project/milne_group/asmith/Projects/2025-07-19-myeloid-specific-enhancer-identification/data/processed/wimm_atlas/2025-11-26-tss-enrichment-scores-all.parquet"
).set_index("path")
tss_scores

,tss_0,tss_1,tss_2,tss_3,tss_4,tss_5,tss_6,tss_7,tss_8,tss_9,...,tss_77580,tss_77581,tss_77582,tss_77583,tss_77584,tss_77585,tss_77586,tss_77587,tss_77588,tss_77589
path,,,,,,,,,,,,,,,,,,,,,
/project/milne_group/asmith/Projects/2025-07-19-myeloid-specific-enhancer-identification/data/processed/wimm_atlas/bigwigs/cell_line_atlas/HCT116_7.bw,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.000000,0.000000,3.291282,6.736002,0.752894,0.000000,0.000000,0.000000,0.000000,35.216371
/project/milne_group/asmith/Projects/2025-07-19-myeloid-specific-enhancer-identification/data/processed/wimm_atlas/bigwigs/cell_line_atlas/Huh7_5.bw,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.000000,0.000000,6.667574,21.265335,5.605069,0.000000,0.000000,0.000000,1.604923,47.890224
/project/milne_group/asmith/Projects/2025-07-19-myeloid-specific-enhancer-identification/data/processed/wimm_atlas/bigwigs/cell_line_atlas/Huh7_3.bw,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,9.749071,0.000000,6.597523,20.418580,17.897798,0.000000,0.000000,0.000000,0.000000,40.133743
/project/milne_group/asmith/Projects/2025-07-19-myeloid-specific-enhancer-identification/data/processed/wimm_atlas/bigwigs/cell_line_atlas/786-O_2.bw,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.000000,3.899533,7.615632,15.603983,5.595073,0.000000,11.140951,0.000000,1.329518,20.153557
/project/milne_group/asmith/Projects/2025-07-19-myeloid-specific-enhancer-identification/data/processed/wimm_atlas/bigwigs/cell_line_atlas/786-O_3.bw,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.000000,0.000000,7.776767,11.050169,5.145597,0.000000,0.000000,0.000000,0.000000,9.661030
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
/project/milne_group/cchahrou/data/2024-10-14_chip_patient/seqnado_output/bigwigs/deeptools/unscaled/CM-863388-1_H3K4me1.bigWig,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.446063,1.442533,0.000000,0.179713,0.369065,0.016852,0.000000,0.440762,0.116252,0.894297
/project/milne_group/cchahrou/data/2025-04-03_atac_erica/seqnado_output_omit/bigwigs/deeptools/unscaled/ATAC-SEM-1.bigWig,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,3.005580,0.590650,6.402047,8.579279,7.829505,1.375905,0.000000,0.000000,3.900333,18.557487
/project/milne_group/cchahrou/data/2025-04-03_atac_erica/seqnado_output_omit/bigwigs/deeptools/unscaled/ATAC-SEM-2.bigWig,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.000000,0.538174,6.628133,8.890012,6.245461,0.000000,2.796611,0.000000,3.387086,19.931284


In [4]:
tss_score_stats = tss_scores.T.describe()
tss_score_stats

path,/project/milne_group/asmith/Projects/2025-07-19-myeloid-specific-enhancer-identification/data/processed/wimm_atlas/bigwigs/cell_line_atlas/HCT116_7.bw,/project/milne_group/asmith/Projects/2025-07-19-myeloid-specific-enhancer-identification/data/processed/wimm_atlas/bigwigs/cell_line_atlas/Huh7_5.bw,/project/milne_group/asmith/Projects/2025-07-19-myeloid-specific-enhancer-identification/data/processed/wimm_atlas/bigwigs/cell_line_atlas/Huh7_3.bw,/project/milne_group/asmith/Projects/2025-07-19-myeloid-specific-enhancer-identification/data/processed/wimm_atlas/bigwigs/cell_line_atlas/786-O_2.bw,/project/milne_group/asmith/Projects/2025-07-19-myeloid-specific-enhancer-identification/data/processed/wimm_atlas/bigwigs/cell_line_atlas/786-O_3.bw,/project/milne_group/asmith/Projects/2025-07-19-myeloid-specific-enhancer-identification/data/processed/wimm_atlas/bigwigs/cell_line_atlas/SW480_1.bw,/project/milne_group/asmith/Projects/2025-07-19-myeloid-specific-enhancer-identification/data/processed/wimm_atlas/bigwigs/cell_line_atlas/FaDu_0.bw,/project/milne_group/asmith/Projects/2025-07-19-myeloid-specific-enhancer-identification/data/processed/wimm_atlas/bigwigs/cell_line_atlas/HK-2_4.bw,/project/milne_group/asmith/Projects/2025-07-19-myeloid-specific-enhancer-identification/data/processed/wimm_atlas/bigwigs/cell_line_atlas/Caco-2_10.bw,/project/milne_group/asmith/Projects/2025-07-19-myeloid-specific-enhancer-identification/data/processed/wimm_atlas/bigwigs/cell_line_atlas/786-O_6.bw,...,/project/milne_group/cchahrou/data/2024-10-14_chip_patient/seqnado_output/bigwigs/deeptools/unscaled/CM-863388-2_PolII.bigWig,/project/milne_group/cchahrou/data/2024-10-14_chip_patient/seqnado_output/bigwigs/deeptools/unscaled/CM-26754-1_H3K27ac.bigWig,/project/milne_group/cchahrou/data/2024-10-14_chip_patient/seqnado_output/bigwigs/deeptools/unscaled/CM-863388-2_CTCF.bigWig,/project/milne_group/cchahrou/data/2024-10-14_chip_patient/seqnado_output/bigwigs/deeptools/unscaled/CM-863388-6_H3K27ac.bigWig,/project/milne_group/cchahrou/data/2024-10-14_chip_patient/seqnado_output/bigwigs/deeptools/unscaled/CM-863388-1_CTCF.bigWig,/project/milne_group/cchahrou/data/2024-10-14_chip_patient/seqnado_output/bigwigs/deeptools/unscaled/CM-863388-1_H3K4me1.bigWig,/project/milne_group/cchahrou/data/2025-04-03_atac_erica/seqnado_output_omit/bigwigs/deeptools/unscaled/ATAC-SEM-1.bigWig,/project/milne_group/cchahrou/data/2025-04-03_atac_erica/seqnado_output_omit/bigwigs/deeptools/unscaled/ATAC-SEM-2.bigWig,/project/milne_group/cchahrou/data/2025-04-03_atac_erica/seqnado_output_omit/bigwigs/deeptools/unscaled/ATAC-SEM-3.bigWig,/project/milne_group/cchahrou/data/2025-04-03_atac_erica/seqnado_output_omit/bigwigs/deeptools/unscaled/ATAC-SEM-4.bigWig
count,7.759000e+04,77590.000000,7.759000e+04,77590.000000,77590.000000,77590.000000,77590.000000,77590.000000,7.759000e+04,77590.000000,...,7.759000e+04,7.759000e+04,7.759000e+04,7.759000e+04,7.759000e+04,77590.000000,7.759000e+04,77590.000000,7.759000e+04,77590.000000
mean,4.642383e+02,169.148545,2.814540e+02,39.810807,66.718372,41.425052,25.905864,26.396307,5.083524e+02,155.253463,...,3.379343e+03,1.388834e+03,2.390614e+05,3.003684e+03,2.879638e+03,0.757222,1.833831e+03,1.978547,8.501009e+02,1.905576
std,2.595818e+04,5680.923912,8.869838e+03,1630.100136,2863.715669,2164.133291,1131.020612,1185.029428,2.287808e+04,6410.173858,...,5.555581e+05,2.846455e+05,1.459498e+07,4.292495e+05,5.732257e+05,1.747769,3.046370e+05,2.764123,2.362530e+05,3.049266
min,0.000000e+00,0.000000,0.000000e+00,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000e+00,0.000000,...,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000,0.000000e+00,0.000000,0.000000e+00,0.000000
25%,0.000000e+00,0.000000,0.000000e+00,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000e+00,0.000000,...,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000,0.000000e+00,0.000000,0.000000e+00,0.000000
50%,0.000000e+00,0.000000,0.0000

In [5]:
# tss_score_stats_plt = (
#     tss_score_stats.T.reset_index()
#     .melt(id_vars="path", var_name="statistic", value_name="value")
#     .assign(
#         sample=lambda df: df["path"].str.split("/").str[-1].str.replace(".parquet", ""),
#         log_value=lambda df: np.log10(df["value"] + 1e-6),
#     )
# )  # Add small constant to avoid log(0 )
# tss_score_stats_plt_mean = tss_score_stats_plt[tss_score_stats_plt["statistic"] == "mean"]

# px.histogram(
#     tss_score_stats_plt_mean,
#     x="log_value",
#     nbins=50,
#     title="Distribution of Mean TSS Enrichment Scores",
# )

# Scaling determination

In [ ]:
# scale_factors = []

# # bw_files_sample = bw_files[:5]

# for bw in tqdm(bw_files, desc="Processing bigwig files", total=len(bw_files)):
#     path = bw.resolve()
#     name = bw.stem
#     cmd = [
#             "/ceph/project/milne_group/asmith/software/cargo/bin/bamnado",
#             "bigwig-infer-scale",
#             "--bigwig",
#             str(path),
#             "--format",
#             "json",
#         ]
#     output = subprocess.run(cmd, check=True, capture_output=True, text=True)
#     result = json.loads(output.stdout)
#     result['samplename'] = name
#     result['path'] = path
#     scale_factors.append(pd.Series(result))

# df_scale_factors = pd.concat(scale_factors, axis=1).T

In [7]:
BAMNADO = "/ceph/project/milne_group/asmith/software/cargo/bin/bamnado"

def infer_scale_factor(bw: Path):
    path = bw.resolve()
    name = bw.stem

    cmd = [
        BAMNADO,
        "bigwig-infer-scale",
        "--bigwig",
        str(path),
        "--format",
        "json",
    ]

    output = subprocess.run(
        cmd,
        check=True,
        capture_output=True,
        text=True,
    )

    result = json.loads(output.stdout)
    result["samplename"] = name
    result["path"] = str(path)

    return pd.Series(result)


# Adjust depending on IO / CPU pressure
max_workers = 16

scale_factors = []

with ThreadPoolExecutor(max_workers=max_workers) as executor:
    futures = [executor.submit(infer_scale_factor, bw) for bw in bw_files]

    for future in tqdm(
        as_completed(futures),
        total=len(futures),
        desc="Processing bigwig files",
    ):
        scale_factors.append(future.result())

df_scale_factors = pd.concat(scale_factors, axis=1).T

Processing bigwig files:   0%|          | 0/934 [00:00<?, ?it/s]

In [8]:
df_scale_factors

,norm_method,scale_factor,library_size,bin_size,min_val,second_min_val,ratio,pseudocount,confident,warnings,chroms_scanned,samplename,path
0,Ambiguous,1.732819,1732818.827278,32,0.577094,1.154189,2.0,None,True,[],2,Caco-2_10,/ceph/project/milne_group/asmith/Projects/2025...
1,Ambiguous,3.237028,3237028.17743,32,0.308925,0.617851,2.0,None,True,[],2,HCT116_7,/ceph/project/milne_group/asmith/Projects/2025...
2,Ambiguous,18.516423,18516423.118881,32,0.054006,0.108012,2.0,None,True,[],2,Caco-2_8,/ceph/project/milne_group/asmith/Projects/2025...
3,Ambiguous,5.991855,5991854.832337,32,0.166893,0.333786,2.0,None,True,[],2,Huh7_5,/ceph/project/milne_group/asmith/Projects/2025...
4,Ambiguous,3.867611,3867611.230428,32,0.258558,0.517115,2.0,None,True,[],2,786-O_6,/ceph/project/milne_group/asmith/Projects/2025...
...,...,...,...,...,...,...,...,...,...,...,...,...,...
929,Rpkm,0.170835,170835040.948969,1,5.8536,11.7072,2.0,None,True,[variable_bin_sizes: mixed bin widths detected...,2,Don002_d11,/ceph/project/milne_group/asmith/Projects/2025...
930,Rpkm,0.127431,127431388.761099,1,7.84736,15.6947,1.999997,None,True,[variable_bin_sizes: mixed bin widths detected...,2,Don003_d17,/ceph/project/milne_group/asmith/Projects/2025...
931,Rpkm,0.162854,162853982.74929,1,6.14047,12.2809,1.999993,None,True,[variable_bin_sizes: mixed bin widths detected...,2,Don003_d11,/ceph/project/milne_group/asmith/Projects/2025...
932,Rpkm,0.129952,129952153.56071,1,7.69514,15.3903,2.000003,None,True,[variable_bin_sizes: mixed bin widths detected...,2,Don002_d17,/ceph/project/milne_group/asmith/Projects/2025...


In [ ]:
# raw_per_bin = normalised_value × scale_factor